# 08 · The pipeline: search, dedup, real energies

`import rxembed.pipeline as rx` adds the optional half. The verb takes a SMILES, an `.xyz` path or a Mol
and returns a chainable `Ensemble`.

- `mc()` runs an openconf torsional search, `prune()` dedups, both in place so they chain
- `lowest`, `representatives`, `align`, `score`, `optimize` return a new ensemble and leave this one alone
- `measure`, `cluster`, `landscape` change nothing

To rank conformers you need a real energy: the force field is meaningless for a metal or a charged TS. The
ladder is `ff -> gfnff -> gfn2 -> gxtb`. `score()` is a single point, `optimize()` relaxes and holds any
frozen core. Energies are kcal/mol and only relative values mean anything. Needs an `xtb` binary.


In [ ]:
import tempfile

import rxembed.pipeline as rx
import xyzrender

rx.set_verbose("INFO")
IBU = "CC(C)Cc1ccc(C(C)C(=O)O)cc1"  # ibuprofen; several real rotamers

## The chain

`embed()` returns one RDKit Mol holding many conformers. `.discarded` keeps what a prune merged away, so
nothing is lost.


In [ ]:
ens = rx.embed("OC(=O)CCCCc1ccccc1").mc().prune()
print(len(ens), "kept,", len(ens.discarded), "merged away")
print(repr(ens))

## The geometry gate

`geom_check.check` is the acceptance test every notebook here calls: broken conjugation, bad H positions,
non-bonded clashes, stretched bonds, a moved frozen core.


In [ ]:
rep = rx.geom_check.check(ens.mol, conf_id=ens.ids[0])
print("clean" if rep.ok() else f"violations: {rep.violations[:4]}")

## The pool: embed → search → prune → keep the lowest few

Build a diverse, deduplicated pool on the cheap FF, then spend the calculator only on the best candidates.

In [ ]:
# stereo="free": ibuprofen's undefined α-stereocentre would split this into an EnsembleSet of two
# enantiomers (identical energies): the tier ladder below is about energy, not stereochemistry.
pool = rx.embed(IBU, stereo="free").mc().prune()
cand = pool.lowest(6)  # the 6 lowest on the force field, to re-rank with a real energy
print(f"{len(pool)} pruned conformers -> re-ranking the lowest {len(cand)}")
clean = [c for c in pool.ids if rx.geom_check.check(pool.mol, conf_id=c).ok()]
print(f"geometry.check: {len(clean)}/{len(pool.ids)} clean: only sane geometries reach a calculator")

## The full ladder: ff → gfnff → gfn2 → g-xTB

`score()` re-ranks on an honest single point without moving atoms; `relative()` gives ΔE from the minimum. Run all four tiers on the same candidates: the FF order is not the g-xTB order, which is the whole point of scoring. Each `score()` returns a fresh ensemble (non-clobbering).

In [ ]:
print("ΔE (kcal/mol) across the tiers on the same 6 candidates:")
print(f"  ff   : {sorted(round(e, 2) for e in cand.relative().values())}")  # the FF pool energies
print(f"  gfnff: {sorted(round(e, 2) for e in cand.score('gfnff').relative().values())}")
print(f"  gfn2 : {sorted(round(e, 2) for e in cand.score('gfn2').relative().values())}")
scored = cand.score("gxtb")  # keep the g-xTB single points (reused in the non-clobber check below)
print(f"  gxtb : {sorted(round(e, 2) for e in scored.relative().values())}")
print("lowest by g-xTB:", scored.lowest(1).ids)

## The recommended pipeline: GFN-FF pool re-rank, then g-xTB on the few

GFN-FF has dispersion + H-bond terms the UFF relax lacks, at ~FF cost: the right tier to rank the *whole pool* (the first NCI-aware energy in the loop), before spending g-xTB on only the survivors. One clear, not-slow call each.

In [ ]:
ranked = pool.score("gfnff").prune()  # NCI-aware pool re-rank
print("GFN-FF ΔE:", sorted(round(e, 1) for e in ranked.relative().values())[:6])
final = ranked.lowest(2).optimize("gxtb", level="loose")  # g-xTB opt, the best few only
print("g-xTB opt ΔE:", sorted(round(e, 2) for e in final.relative().values()))
from xyzrender import load

xyzrender.render(
    load(final.lowest(2).dump(tempfile.mktemp(suffix=".xyz")), ensemble=True, ensemble_color="spectral"), no_hy=True
)

## Honest energies, non-clobbering

`score('gxtb')` / `optimize('gxtb')` raise if the calculator produces nothing: never a silent FF fallback the caller mistakes for xTB. g-xTB has no ALPB, so solvent comes from a GFN2 thermodynamic-cycle correction, `E_gxtb(gas) + [E_gfn2(solv) − E_gfn2(gas)]`, a real solvated energy, not a silent gas-phase. And they return new ensembles: the input geometry is never moved under you.

In [ ]:
print("input pool untouched:", len(pool), "conformers on the FF path")
print("scored / optimised are independent ensembles:", len(scored), "/", len(final))
# g-xTB in solvent: no ALPB, so a GFN2 thermodynamic-cycle correction; runs, never a raise or silent gas-phase
wet = cand.lowest(2).score("gxtb", solvent="water")
print(f"g-xTB/water on {len(wet)} conf: ΔE {sorted(round(e, 2) for e in wet.relative().values())} (solvent-corrected)")